<a href="https://colab.research.google.com/github/Malik-Nauman-Rauf-207/control-systems-book/blob/main/CIMID_Chapter1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐾 Chapter 1 Lab: The Secret Code of Everything

Welcome to the interactive companion lab for **Chapter 1**!

Here, you will help our guide, **Vector the Mathematical Cat**, explore the hidden control systems all around us. By adjusting the sliders in these simulations, you will see exactly how feedback loops react to disturbances in real-time.

---
### Step 1: Run the Setup
Before we begin, run the code cell below to load the interactive plotting libraries. *(Click the "Play" button on the left of the cell).*

In [2]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Set a cleaner style for our plots
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Setup complete! Vector is ready to go.")

✅ Setup complete! Vector is ready to go.


---
## 1. Biology: The Feline Thermostat 🌡️

Vector’s body needs to stay near a healthy **101.5°F**. When a disturbance (the weather) tries to change that, his biological controller (his brain) activates an actuator (shivering or sweating) to fight back.

**Your Mission:** Use the slider to change the outside weather. Watch how hard Vector's body has to work to balance heat loss with internal heat generation! The greater the difference between purrfect body temperature, the harder Vector has to work.

In [ ]:
@widgets.interact(outside_temp=widgets.IntSlider(min=-20, max=100, step=5, value=30, description='Outside (°F):'))
def bio_plot(outside_temp):
    times = list(range(60))
    body_temp = [101.5]

    for t in range(1, 60):
        # The Disturbance
        heat_loss = (body_temp[-1] - outside_temp) * 0.05
        # The Feedback Controller
        shivering = (101.5 - body_temp[-1]) * 0.25

        new_temp = body_temp[-1] - heat_loss + shivering
        body_temp.append(new_temp)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(times, body_temp, color='#FF9A3C', linewidth=3, label="Vector's Core Temp")
    ax.axhline(101.5, color='#2D8032', linestyle='--', label='Purr-fect (101.5°F)')
    ax.set_ylim(85, 110)
    ax.set_title("Biological Feedback Loop", fontsize=14, fontweight='bold')
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Temperature (°F)")
    ax.legend()
    plt.show()

interactive(children=(IntSlider(value=30, description='Outside (°F):', min=-20, step=5), Output()), _dom_class…

## 2. Aerospace: Balancing a Falling Skyscraper 🚀

A tall rocket descending toward the ground is like trying to balance a broomstick on your fingertip. Real rockets are heavy and have momentum, so reacting *only* to the tilt would cause the rocket to swing wildly like a pendulum. Instead, the flight computer uses **Proportional-Derivative (PD) control** to measure both the tilt angle and how fast it's swinging.

**Your Mission:** A heavy wind gust hits the rocket between 3 and 4 seconds. Adjust the **Thruster Power** to see how the system reacts.
* *Test it at 0.5:* The engine doesn't push back hard enough, and the wind blows the rocket past the red crash line.
* *Test it at 3.0:* A smooth, efficient recovery back to perfectly upright.
* *Test it at 15.0:* The controller overreacts! Notice in the bottom graph how the engine maxes out back and forth, vibrating the rocket violently and wasting massive amounts of fuel.

In [ ]:
import numpy as np

@widgets.interact(thruster=widgets.FloatSlider(min=0.5, max=15.0, step=0.5, value=3.0, description='Thruster:'))
def rocket_plot(thruster):
    # High-resolution time array for smooth physics
    times = np.linspace(0, 25, 300)
    dt = times[1] - times[0]

    tilt = [0]
    velocity = 0
    thruster_effort = [0]

    for t in times[1:]:
        # Wind gust hits between 3 and 4 seconds
        wind_force = 45.0 if 3 <= t <= 4 else 0.0

        # The Control System (PD Controller)
        current_tilt = tilt[-1]
        command = (thruster * current_tilt) + (0.8 * velocity)

        # Physics Engine (Momentum & Acceleration)
        acceleration = wind_force - command
        velocity += acceleration * dt
        new_tilt = current_tilt + velocity * dt

        tilt.append(new_tilt)
        thruster_effort.append(command)

    # Create a 2-panel layout
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios': [2, 1]})

    # TOP PLOT: Rocket Tilt
    ax1.plot(times, tilt, color='purple', linewidth=3)
    ax1.axhline(20, color='red', linestyle='--', label='Crash Limit (+20°)')
    ax1.axhline(-20, color='red', linestyle='--', label='Crash Limit (-20°)')
    ax1.axhline(0, color='green', linestyle=':', label='Perfectly Upright')
    ax1.set_ylim(-30, 30)
    ax1.set_ylabel("Tilt Angle (Degrees)")
    ax1.set_title("Rocket Landing Dynamics (PD Control)", fontsize=14, fontweight='bold')
    ax1.legend(loc="upper right")

    # BOTTOM PLOT: Thruster Fuel/Effort
    ax2.plot(times, thruster_effort, color='orange', linewidth=2, label="Engine Thrust (Fuel)")
    ax2.fill_between(times, thruster_effort, color='orange', alpha=0.3)
    ax2.axhline(0, color='gray', linewidth=1)
    ax2.set_xlabel("Time (seconds)")
    ax2.set_ylabel("Thrust Power")
    ax2.set_ylim(-60, 60)
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

---
## 3. Automotive: The Invisible Bumper 🚗

Adaptive cruise control tries to maintain a safe distance behind the vehicle ahead. The controller measures the error between the cars and decides how hard to apply the throttle or brakes.

**Your Mission:** Vector is driving the blue car at 60 mph. The lead car (red) suddenly hits the brakes at 20 seconds. Adjust the **Reaction Speed** to find a safe, smooth braking curve.

In [3]:
@widgets.interact(reaction=widgets.FloatSlider(min=0.1, max=1.5, step=0.1, value=0.4, description='Reaction:'))
def cruise_plot(reaction):
    times = list(range(60))
    lead_speed = [60 if t < 20 else 40 for t in times]
    our_speed = [60]

    for t in range(1, 60):
        error = lead_speed[t-1] - our_speed[-1]
        our_speed.append(our_speed[-1] + (error * reaction))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(times, lead_speed, color='red', linestyle='--', linewidth=3, label='Lead Car (Brakes Applied)')
    ax.plot(times, our_speed, color='#0077CC', linewidth=3, label="Vector's Car")
    ax.set_ylim(20, 80)
    ax.set_title("Adaptive Cruise Control", fontsize=14, fontweight='bold')
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Speed (mph)")
    ax.legend()
    plt.show()

interactive(children=(FloatSlider(value=0.4, description='Reaction:', max=1.5, min=0.1), Output()), _dom_class…

---
## 4. Environmental: The Smart Greenhouse 🪴

Plants don't care if their gardener is a mathematical cat. A smart greenhouse measures soil moisture and turns on the water pump when things get too dry. This is called **on-off control** (or "bang-bang" control).

**Your Mission:** The hot sun is constantly evaporating water. Adjust the **Pump Power** to keep the moisture cycling comfortably around the 50% target.

In [ ]:
@widgets.interact(pump=widgets.IntSlider(min=1, max=15, step=1, value=8, description='Pump Power:'))
def farm_plot(pump):
    times = list(range(100))
    moisture = [50]

    for t in range(1, 100):
        current = moisture[-1]
        sun_dry = 2  # The sun constantly removes 2% moisture

        # Sprinkler turns on ONLY if moisture drops below 40%
        water_added = pump if current < 40 else 0
        moisture.append(current - sun_dry + water_added)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(times, moisture, color='teal', linewidth=2, label="Measured Moisture")
    ax.axhline(50, color='green', linestyle='--', label='Perfect Moisture (50%)')
    ax.set_ylim(0, 100)
    ax.set_title("Bang-Bang Moisture Controller", fontsize=14, fontweight='bold')
    ax.set_xlabel("Time")
    ax.set_ylabel("Soil Moisture (%)")
    ax.legend()
    plt.show()

---
## 5. The Wildcard: Video Game Cameras 🎮

When Vector plays a 3D game, the camera follows his character like it's attached by an invisible rubber band. If it snapped instantly to the player, the game would be too jerky!

**Your Mission:** Vector's character teleports 50 feet forward. Adjust the **Camera Stiffness** to see the trade-off between a "snappy" action-game camera and a "smooth" exploration-game camera.

In [ ]:
@widgets.interact(stiffness=widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.15, description='Stiffness:'))
def cam_plot(stiffness):
    times = list(range(40))
    player_pos = [0 if t < 5 else 50 for t in times] # Player jumps ahead at t=5
    camera_pos = [0]

    for t in range(1, 40):
        error = player_pos[t] - camera_pos[-1]
        camera_pos.append(camera_pos[-1] + (error * stiffness))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(times, player_pos, color='red', linestyle='--', linewidth=2, label='Player Position')
    ax.plot(times, camera_pos, color='#0077CC', linewidth=3, label='Camera Tracking')
    ax.set_title("Proportional Camera Tracking", fontsize=14, fontweight='bold')
    ax.set_xlabel("Frames")
    ax.set_ylabel("Virtual Distance")
    ax.legend()
    plt.show()

---
> *"A smart system does not merely act. It watches what happens after it acts, then learns what to do next."* — **Vector** 🐾

**Great job!** You've successfully interacted with five different control loops. You are now ready to head back to the book and begin **Chapter 2**.